# Questão 9 - Sistema de recomendação baseado em ontologia

## 1. Domínio

O domínio escolhido é a recomendação de destinos turísticos no Brasil. O usuário informa seus interesses, o orçamento por dia e o clima que prefere, e o sistema indica os destinos mais adequados, explicando o motivo de cada indicação.

A ontologia foi escrita em OWL 2 com a biblioteca Owlready2 e o raciocínio é feito pelo HermiT, que precisa do Java instalado. Os destinos são reais, mas os custos e as avaliações são valores aproximados usados apenas como exemplo.

In [1]:
import types

import pandas as pd
from owlready2 import (
    AllDifferent, AllDisjoint, ConstrainedDatatype, DataProperty, FunctionalProperty, ObjectProperty,
    OneOf, OwlReadyInconsistentOntologyError, PropertyChain, SymmetricProperty, Thing,
    TransitiveProperty, World, destroy_entity, sync_reasoner,
)

mundo = World()
onto = mundo.get_ontology("http://exemplo.org/turismo.owl")

## 2. Classes

A ontologia tem quatro grupos de classes.

**Lugar:** Destino, Estado e Regiao. Um destino fica em um estado e um estado fica em uma região. Os estados são identificados pela sigla.

**Atracao:** o que o destino oferece. Divide-se em AtracaoNatural (Praia, Cachoeira, Trilha) e AtracaoCultural (Museu, CentroHistorico).

**Interesse e Clima:** valores usados para descrever tanto os destinos quanto as preferências do viajante.

**Viajante:** o usuário do sistema.

Além dessas, existem classes definidas, que não recebem indivíduos manualmente. Elas têm uma condição necessária e suficiente e é o reasoner que decide quem pertence a elas: DestinoDePraia, DestinoCultural, DestinoDeNatureza, DestinoEconomico, DestinoCompleto e ViajanteEconomico.

In [2]:
with onto:
    class Lugar(Thing): pass
    class Destino(Lugar): pass
    class Estado(Lugar): pass
    class Regiao(Lugar): pass

    class Atracao(Thing): pass
    class AtracaoNatural(Atracao): pass
    class AtracaoCultural(Atracao): pass
    class Praia(AtracaoNatural): pass
    class Cachoeira(AtracaoNatural): pass
    class Trilha(AtracaoNatural): pass
    class Museu(AtracaoCultural): pass
    class CentroHistorico(AtracaoCultural): pass

    class Interesse(Thing): pass
    class Clima(Thing): pass
    class Viajante(Thing): pass

    AllDisjoint([Lugar, Atracao, Interesse, Clima, Viajante])
    AllDisjoint([Destino, Estado, Regiao])
    AllDisjoint([AtracaoNatural, AtracaoCultural])

## 3. Propriedades

São 11 propriedades de objeto e 6 de dados.

**Propriedades de objeto**

temAtracao e atracaoDe: ligam destino e atração. Uma é a inversa da outra.

localizadoEm e contem: ligam um lugar ao lugar maior onde ele fica. localizadoEm é transitiva, então um destino que fica em AL (Alagoas) também fica no Nordeste.

temClima: clima do destino. É funcional, cada destino tem um só.

atendeInteresse: liga uma atração ao interesse que ela satisfaz.

ofereceInteresse: liga o destino aos interesses que ele satisfaz. Ninguém preenche essa propriedade. Ela é deduzida por uma cadeia de propriedades: se o destino tem uma atração e a atração atende um interesse, então o destino oferece esse interesse.

proximoDe: destinos próximos. É simétrica.

temInteresse, prefereClima e jaVisitou: descrevem o viajante.

**Propriedades de dados**

custoDiario, avaliacao e diasSugeridos descrevem o destino. orcamentoDiario e diasDisponiveis descrevem o viajante. nome é usado para exibição.

In [3]:
with onto:
    class temAtracao(ObjectProperty):
        domain = [Destino]
        range = [Atracao]

    class atracaoDe(ObjectProperty):
        inverse_property = temAtracao

    class localizadoEm(ObjectProperty, TransitiveProperty):
        domain = [Lugar]
        range = [Lugar]

    class contem(ObjectProperty):
        inverse_property = localizadoEm

    class temClima(ObjectProperty, FunctionalProperty):
        domain = [Destino]
        range = [Clima]

    class atendeInteresse(ObjectProperty):
        domain = [Atracao]
        range = [Interesse]

    class ofereceInteresse(ObjectProperty):
        domain = [Destino]
        range = [Interesse]

    class proximoDe(ObjectProperty, SymmetricProperty):
        domain = [Destino]
        range = [Destino]

    class temInteresse(ObjectProperty):
        domain = [Viajante]
        range = [Interesse]

    class prefereClima(ObjectProperty):
        domain = [Viajante]
        range = [Clima]

    class jaVisitou(ObjectProperty):
        domain = [Viajante]
        range = [Destino]

    class custoDiario(DataProperty, FunctionalProperty):
        domain = [Destino]
        range = [int]

    class avaliacao(DataProperty, FunctionalProperty):
        domain = [Destino]
        range = [float]

    class diasSugeridos(DataProperty, FunctionalProperty):
        domain = [Destino]
        range = [int]

    class orcamentoDiario(DataProperty, FunctionalProperty):
        domain = [Viajante]
        range = [int]

    class diasDisponiveis(DataProperty, FunctionalProperty):
        domain = [Viajante]
        range = [int]

    class nome(DataProperty, FunctionalProperty):
        range = [str]

    ofereceInteresse.property_chain.append(PropertyChain([temAtracao, atendeInteresse]))

## 4. Indivíduos de apoio, restrições e axiomas

Primeiro são criados os interesses e os climas. Depois entram as restrições.

**Restrições de valor:** toda praia atende o interesse praia e o interesse natureza, toda trilha atende aventura e natureza, e assim por diante. Com isso não é preciso informar o interesse de cada atração, o reasoner deduz.

**Restrições sobre Destino:** todo destino tem exatamente um clima e fica em algum estado.

**Classes definidas:** DestinoDePraia é um destino que tem alguma praia. DestinoEconomico é um destino com custo diário de até 300 reais. DestinoCompleto é um destino que tem atração natural e atração cultural.

In [4]:
with onto:
    praia, cultura, aventura, natureza, historia = [Interesse(n) for n in ("praia", "cultura", "aventura", "natureza", "historia")]
    quente, ameno, frio = [Clima(n) for n in ("quente", "ameno", "frio")]
    AllDifferent([praia, cultura, aventura, natureza, historia])
    AllDifferent([quente, ameno, frio])

    Praia.is_a += [atendeInteresse.value(praia), atendeInteresse.value(natureza)]
    Cachoeira.is_a += [atendeInteresse.value(natureza), atendeInteresse.value(aventura)]
    Trilha.is_a += [atendeInteresse.value(aventura), atendeInteresse.value(natureza)]
    Museu.is_a += [atendeInteresse.value(cultura)]
    CentroHistorico.is_a += [atendeInteresse.value(historia), atendeInteresse.value(cultura)]

    Destino.is_a += [temClima.exactly(1, Clima), localizadoEm.some(Estado)]
    Estado.is_a += [localizadoEm.some(Regiao)]

    class DestinoDePraia(Destino):
        equivalent_to = [Destino & temAtracao.some(Praia)]

    class DestinoCultural(Destino):
        equivalent_to = [Destino & temAtracao.some(AtracaoCultural)]

    class DestinoDeNatureza(Destino):
        equivalent_to = [Destino & temAtracao.some(Cachoeira | Trilha)]

    class DestinoCompleto(Destino):
        equivalent_to = [Destino & temAtracao.some(AtracaoNatural) & temAtracao.some(AtracaoCultural)]

    class DestinoEconomico(Destino):
        equivalent_to = [Destino & custoDiario.some(ConstrainedDatatype(int, max_inclusive=300))]

    class ViajanteEconomico(Viajante):
        equivalent_to = [Viajante & orcamentoDiario.some(ConstrainedDatatype(int, max_inclusive=300))]

## 5. Base de conhecimento

Os destinos são cadastrados apenas como Destino, com suas atrações, estado, clima e custo. Nenhum deles é marcado como destino de praia, cultural ou econômico. Essa classificação fica para o reasoner.

In [5]:
regioes = {"Nordeste": ["AL", "PE", "BA"], "Sudeste": ["MG", "RJ"], "Sul": ["RS", "PR"], "Centro_Oeste": ["MS"]}

tipos = {"praia": Praia, "cachoeira": Cachoeira, "trilha": Trilha, "museu": Museu, "centro": CentroHistorico}

# destino: (estado, clima, custo por dia, avaliação, dias, atrações)
destinos = {
    "Maragogi": ("AL", quente, 380, 4.7, 4, {"praia": "Piscinas_Naturais_de_Maragogi"}),
    "Porto_de_Galinhas": ("PE", quente, 420, 4.6, 4, {"praia": "Praia_de_Muro_Alto"}),
    "Olinda": ("PE", quente, 200, 4.4, 2, {"centro": "Sitio_Historico_de_Olinda", "museu": "Museu_do_Mamulengo"}),
    "Salvador": ("BA", quente, 300, 4.5, 4, {"praia": "Praia_do_Porto_da_Barra", "centro": "Pelourinho", "museu": "Museu_Afro_Brasileiro"}),
    "Lencois": ("BA", ameno, 220, 4.8, 5, {"cachoeira": "Cachoeira_da_Fumaca", "trilha": "Trilha_do_Vale_do_Pati"}),
    "Ouro_Preto": ("MG", ameno, 250, 4.7, 3, {"centro": "Centro_Historico_de_Ouro_Preto", "museu": "Museu_da_Inconfidencia"}),
    "Rio_de_Janeiro": ("RJ", quente, 500, 4.6, 5, {"praia": "Praia_de_Ipanema", "trilha": "Trilha_da_Pedra_da_Gavea", "museu": "Museu_do_Amanha"}),
    "Gramado": ("RS", frio, 480, 4.5, 3, {"museu": "Museu_do_Festival_de_Cinema"}),
    "Foz_do_Iguacu": ("PR", ameno, 320, 4.8, 3, {"cachoeira": "Cataratas_do_Iguacu", "trilha": "Trilha_das_Cataratas"}),
    "Bonito": ("MS", quente, 450, 4.9, 4, {"cachoeira": "Cachoeiras_da_Boca_da_Onca", "trilha": "Trilha_do_Rio_Sucuri"}),
}

with onto:
    for regiao, estados in regioes.items():
        r = Regiao(regiao)
        for e in estados:
            Estado(e).localizadoEm = [r]

    for d, (estado, clima, custo, nota, dias, atracoes) in destinos.items():
        ind = Destino(d, nome=d.replace("_", " "), temClima=clima, custoDiario=custo, avaliacao=nota, diasSugeridos=dias)
        ind.localizadoEm = [onto[estado]]
        ind.temAtracao = [tipos[t](a, nome=a.replace("_", " ")) for t, a in atracoes.items()]

    onto.Maragogi.proximoDe = [onto.Porto_de_Galinhas]
    onto.Olinda.proximoDe = [onto.Porto_de_Galinhas]

    AllDifferent(list(Destino.instances()))
    AllDifferent(list(Estado.instances()))
    AllDifferent(list(Regiao.instances()))

print("Classes:", len(list(onto.classes())))
print("Propriedades de objeto:", len(list(onto.object_properties())))
print("Propriedades de dados:", len(list(onto.data_properties())))
print("Indivíduos:", len(list(onto.individuals())))

Classes: 21
Propriedades de objeto: 11
Propriedades de dados: 6
Indivíduos: 49


In [6]:
onto.save(file="turismo.owl", format="rdfxml")

## 6. Raciocínio com o HermiT

Antes de rodar o reasoner, é guardado o que foi declarado para alguns destinos. Depois os mesmos dados são mostrados de novo para comparar.

In [7]:
def nomes(itens):
    return ", ".join(sorted(i.name for i in itens)) or "-"


def retrato():
    return pd.DataFrame(
        {
            d.name: {
                "classes": nomes(c for c in d.is_a if hasattr(c, "name")),
                "localizadoEm": nomes(d.localizadoEm),
                "ofereceInteresse": nomes(d.ofereceInteresse),
                "proximoDe": nomes(d.proximoDe),
            }
            for d in (onto.Maragogi, onto.Salvador, onto.Lencois, onto.Porto_de_Galinhas)
        }
    ).T


pd.set_option("display.max_colwidth", None)
antes = retrato()
antes

,classes,localizadoEm,ofereceInteresse,proximoDe
Maragogi,Destino,AL,-,Porto_de_Galinhas
Salvador,Destino,BA,-,-
Lencois,Destino,BA,-,-
Porto_de_Galinhas,Destino,PE,-,"Maragogi, Olinda"


In [8]:
with onto:
    sync_reasoner(mundo, infer_property_values=True, debug=0)

print("Ontologia consistente. Classes insatisfatíveis:", list(mundo.inconsistent_classes()))
retrato()

Ontologia consistente. Classes insatisfatíveis: []


,classes,localizadoEm,ofereceInteresse,proximoDe
Maragogi,DestinoDePraia,"AL, Nordeste","natureza, praia",Porto_de_Galinhas
Salvador,"DestinoCompleto, DestinoDePraia, DestinoEconomico","BA, Nordeste","cultura, historia, natureza, praia",-
Lencois,"DestinoDeNatureza, DestinoEconomico","BA, Nordeste","aventura, natureza",-
Porto_de_Galinhas,DestinoDePraia,"Nordeste, PE","natureza, praia","Maragogi, Olinda"


**Classificação automática de indivíduos:** Maragogi e Porto de Galinhas viraram DestinoDePraia. Salvador virou DestinoDePraia, DestinoCultural, DestinoCompleto e DestinoEconomico. Lençóis virou DestinoDeNatureza e DestinoEconomico. Nada disso foi declarado.

**Descoberta de relacionamentos implícitos:** localizadoEm ganhou a região, pela transitividade. ofereceInteresse foi preenchida pela cadeia de propriedades junto com as restrições de valor. Porto de Galinhas ficou próximo de Maragogi e de Olinda, pela simetria.

A coluna de classes mostra só as mais específicas. DestinoCompleto, por exemplo, já implica DestinoCultural, e essa relação entre as duas classes também foi descoberta pelo reasoner.

A tabela abaixo mostra a classificação de todos os destinos.

In [9]:
definidas = [onto.DestinoDePraia, onto.DestinoCultural, onto.DestinoDeNatureza, onto.DestinoCompleto, onto.DestinoEconomico]
pd.DataFrame(
    {c.name: {d.name: "x" if d in c.instances() else "" for d in onto.Destino.instances()} for c in definidas}
)

,DestinoDePraia,DestinoCultural,DestinoDeNatureza,DestinoCompleto,DestinoEconomico
Maragogi,x,,,,
Porto_de_Galinhas,x,,,,
Olinda,,x,,,x
Salvador,x,x,,x,x
Lencois,,,x,,x
Ouro_Preto,,x,,,x
Rio_de_Janeiro,x,x,x,x,
Gramado,,x,,,
Foz_do_Iguacu,,,x,,
Bonito,,,x,,


## 7. Consultas semânticas

As consultas são feitas em SPARQL sobre a ontologia já com as inferências. As três abaixo só funcionam por causa do reasoner, porque usam fatos que não foram declarados.

**Consulta 1:** destinos do Nordeste que oferecem o interesse cultura.

In [10]:
PREFIXO = "PREFIX t: <http://exemplo.org/turismo.owl#>\n"


def sparql(consulta):
    linhas = mundo.sparql(PREFIXO + consulta)
    return [[getattr(v, "name", v) for v in linha] for linha in linhas]


sparql("""
SELECT ?d ?custo WHERE {
    ?d a t:Destino ; t:localizadoEm t:Nordeste ; t:ofereceInteresse t:cultura ; t:custoDiario ?custo .
} ORDER BY ?custo
""")

[['Olinda', 200], ['Salvador', 300]]

**Consulta 2:** destinos econômicos que são de natureza.

In [11]:
sparql("SELECT ?d WHERE { ?d a t:DestinoEconomico . ?d a t:DestinoDeNatureza . }")

[['Lencois']]

**Consulta 3:** quantos destinos cada região contém.

In [12]:
sparql("""
SELECT ?r (COUNT(?d) AS ?n) WHERE { ?d a t:Destino ; t:localizadoEm ?r . ?r a t:Regiao . }
GROUP BY ?r ORDER BY DESC(?n)
""")

[['Nordeste', 5], ['Sudeste', 2], ['Sul', 2], ['Centro_Oeste', 1]]

## 8. Verificação de consistência

Para mostrar o reasoner detectando um erro, a ontologia salva é carregada em um mundo separado e recebe um fato errado: o indivíduo Pelourinho, que é um CentroHistorico, é declarado também como Praia. Como AtracaoNatural e AtracaoCultural são disjuntas, a ontologia fica inconsistente.

In [13]:
teste = World()
onto_teste = teste.get_ontology("turismo.owl").load()
onto_teste.Pelourinho.is_a.append(onto_teste.Praia)

try:
    sync_reasoner(teste, debug=0)
    print("Consistente")
except OwlReadyInconsistentOntologyError:
    print("Inconsistência detectada: Pelourinho não pode ser atração natural e cultural ao mesmo tempo.")

Inconsistência detectada: Pelourinho não pode ser atração natural e cultural ao mesmo tempo.


## 9. Mecanismo de recomendação

A recomendação é feita em quatro passos.

**Passo 1:** o perfil informado vira um indivíduo da classe Viajante, com seus interesses, orçamento e clima preferido.

**Passo 2:** é criada uma classe definida só para esse viajante. Ela descreve os destinos compatíveis: os que oferecem pelo menos um dos interesses dele e cujo custo diário cabe no orçamento.

**Passo 3:** o reasoner é executado. Ele classifica o viajante (por exemplo, como ViajanteEconomico) e coloca na nova classe os destinos compatíveis. A seleção dos candidatos é feita pelo reasoner e não por um filtro em Python.

**Passo 4:** os candidatos são ordenados por uma pontuação e cada um é mostrado com os fatos que levaram à sua indicação.

A pontuação soma 2 pontos por interesse atendido, 1,5 ponto se o clima for o preferido, 1 ponto se o destino for classificado como DestinoCompleto e a avaliação dividida por 5. Destinos já visitados são retirados.

A retirada dos visitados é feita em Python. Em OWL vale a hipótese do mundo aberto: o fato de não estar escrito que o viajante visitou um destino não permite concluir que ele não visitou.

In [14]:
def recomendar(nome_viajante, interesses, orcamento, clima=None, visitados=(), top=3):
    with onto:
        v = onto.Viajante(nome_viajante, orcamentoDiario=orcamento)
        v.temInteresse = [onto[i] for i in interesses]
        v.prefereClima = [onto[clima]] if clima else []
        v.jaVisitou = [onto[d] for d in visitados]

        compativel = types.new_class(f"DestinoCompativelCom_{nome_viajante}", (onto.Destino,))
        compativel.equivalent_to = [
            onto.Destino
            & onto.ofereceInteresse.some(OneOf(v.temInteresse))
            & onto.custoDiario.some(ConstrainedDatatype(int, max_inclusive=orcamento))
        ]
        sync_reasoner(mundo, infer_property_values=True, debug=0)

    perfil = nomes(c for c in v.is_a if c is not onto.Viajante)
    print(f"Viajante: {nome_viajante} | interesses: {', '.join(interesses)} | orçamento: R$ {orcamento}/dia | clima: {clima or 'indiferente'}")
    print(f"Classificação do viajante pelo reasoner: {perfil}")
    candidatos = [d for d in compativel.instances() if d not in v.jaVisitou]
    print(f"Destinos compatíveis segundo o reasoner: {nomes(compativel.instances())}")
    if v.jaVisitou:
        print(f"Retirados por já terem sido visitados: {nomes(v.jaVisitou)}")

    resultado = []
    for d in candidatos:
        atendidos = [i for i in v.temInteresse if i in d.ofereceInteresse]
        clima_ok = bool(v.prefereClima) and d.temClima in v.prefereClima
        pontos = 2 * len(atendidos) + 1.5 * clima_ok + (d in onto.DestinoCompleto.instances()) + d.avaliacao / 5
        resultado.append((pontos, d, atendidos, clima_ok))
    resultado.sort(key=lambda r: r[0], reverse=True)

    for pos, (pontos, d, atendidos, clima_ok) in enumerate(resultado[:top], 1):
        print(f"\n{pos}. {d.nome} (pontuação {pontos:.2f})")
        print(f"   Localização: {nomes(d.localizadoEm)}")
        print(f"   Classes inferidas: {nomes(c for c in definidas if d in c.instances())}")
        for i in atendidos:
            origem = [a.nome for a in d.temAtracao if i in a.atendeInteresse]
            print(f"   Interesse {i.name}: atendido por {', '.join(origem)}")
        print(f"   Custo: R$ {d.custoDiario}/dia, dentro do orçamento de R$ {orcamento}")
        print(f"   Clima: {d.temClima.name}" + (" (o preferido)" if clima_ok else ""))
        print(f"   Avaliação: {d.avaliacao} | dias sugeridos: {d.diasSugeridos}")

    if not resultado:
        print("\nNenhum destino atende ao perfil informado.")

    destroy_entity(compativel)
    return [d.name for _, d, _, _ in resultado]

**Recomendação 1:** viajante com pouco orçamento que gosta de história e cultura.

In [15]:
_ = recomendar("Ana", ["historia", "cultura"], 300, clima="ameno")

Viajante: Ana | interesses: historia, cultura | orçamento: R$ 300/dia | clima: ameno
Classificação do viajante pelo reasoner: ViajanteEconomico
Destinos compatíveis segundo o reasoner: Olinda, Ouro_Preto, Salvador

1. Ouro Preto (pontuação 6.44)
   Localização: MG, Sudeste
   Classes inferidas: DestinoCultural, DestinoEconomico
   Interesse historia: atendido por Centro Historico de Ouro Preto
   Interesse cultura: atendido por Centro Historico de Ouro Preto, Museu da Inconfidencia
   Custo: R$ 250/dia, dentro do orçamento de R$ 300
   Clima: ameno (o preferido)
   Avaliação: 4.7 | dias sugeridos: 3

2. Salvador (pontuação 5.90)
   Localização: BA, Nordeste
   Classes inferidas: DestinoCompleto, DestinoCultural, DestinoDePraia, DestinoEconomico
   Interesse historia: atendido por Pelourinho
   Interesse cultura: atendido por Pelourinho, Museu Afro Brasileiro
   Custo: R$ 300/dia, dentro do orçamento de R$ 300
   Clima: quente
   Avaliação: 4.5 | dias sugeridos: 4

3. Olinda (pontuação 

**Recomendação 2:** viajante que quer praia e aventura, com orçamento maior, e que já conhece o Rio de Janeiro.

In [16]:
_ = recomendar("Bruno", ["praia", "aventura"], 500, clima="quente", visitados=["Rio_de_Janeiro"])

Viajante: Bruno | interesses: praia, aventura | orçamento: R$ 500/dia | clima: quente
Classificação do viajante pelo reasoner: -
Destinos compatíveis segundo o reasoner: Bonito, Foz_do_Iguacu, Lencois, Maragogi, Porto_de_Galinhas, Rio_de_Janeiro, Salvador
Retirados por já terem sido visitados: Rio_de_Janeiro

1. Salvador (pontuação 5.40)
   Localização: BA, Nordeste
   Classes inferidas: DestinoCompleto, DestinoCultural, DestinoDePraia, DestinoEconomico
   Interesse praia: atendido por Praia do Porto da Barra
   Custo: R$ 300/dia, dentro do orçamento de R$ 500
   Clima: quente (o preferido)
   Avaliação: 4.5 | dias sugeridos: 4

2. Bonito (pontuação 4.48)
   Localização: Centro_Oeste, MS
   Classes inferidas: DestinoDeNatureza
   Interesse aventura: atendido por Cachoeiras da Boca da Onca, Trilha do Rio Sucuri
   Custo: R$ 450/dia, dentro do orçamento de R$ 500
   Clima: quente (o preferido)
   Avaliação: 4.9 | dias sugeridos: 4

3. Maragogi (pontuação 4.44)
   Localização: AL, Nordest

**Recomendação 3:** viajante que quer natureza com orçamento baixo.

In [17]:
_ = recomendar("Carla", ["natureza"], 250)

Viajante: Carla | interesses: natureza | orçamento: R$ 250/dia | clima: indiferente
Classificação do viajante pelo reasoner: ViajanteEconomico
Destinos compatíveis segundo o reasoner: Lencois

1. Lencois (pontuação 2.96)
   Localização: BA, Nordeste
   Classes inferidas: DestinoDeNatureza, DestinoEconomico
   Interesse natureza: atendido por Cachoeira da Fumaca, Trilha do Vale do Pati
   Custo: R$ 220/dia, dentro do orçamento de R$ 250
   Clima: ameno
   Avaliação: 4.8 | dias sugeridos: 5


**Recomendação 4:** perfil sem nenhum destino compatível.

In [18]:
_ = recomendar("Davi", ["praia"], 150)

Viajante: Davi | interesses: praia | orçamento: R$ 150/dia | clima: indiferente
Classificação do viajante pelo reasoner: ViajanteEconomico
Destinos compatíveis segundo o reasoner: -

Nenhum destino atende ao perfil informado.


## 10. Entrada pelo teclado

A função abaixo recebe o perfil pelo teclado e chama a recomendação. Ela não é executada automaticamente para o notebook poder rodar de uma vez.

In [19]:
def recomendar_interativo():
    nome_v = input("Nome: ").strip() or "Visitante"
    print("Interesses disponíveis:", nomes(onto.Interesse.instances()))
    interesses = [i.strip() for i in input("Interesses (separados por vírgula): ").split(",") if i.strip()]
    orcamento = int(input("Orçamento por dia (R$): "))
    clima = input("Clima preferido (quente, ameno, frio ou vazio): ").strip() or None
    return recomendar(nome_v, interesses, orcamento, clima)

## 11. Análise crítica

**Justificativa das principais classes e propriedades**

Destino e Atracao são o centro do modelo, porque a recomendação depende do que cada lugar oferece. A separação entre AtracaoNatural e AtracaoCultural permite definir tipos de destino sem listar cada atração.

Interesse faz a ponte entre o viajante e o destino. O viajante tem interesses e as atrações atendem interesses. A propriedade ofereceInteresse, deduzida por cadeia, junta as duas pontas.

As classes definidas existem para que o reasoner trabalhe. Ao cadastrar um destino novo com uma praia, ele passa a ser DestinoDePraia e a oferecer os interesses praia e natureza sem nenhuma outra mudança.

**Vantagens**

O conhecimento fica separado do código. Para incluir um destino ou um tipo de atração basta alterar a ontologia.

As recomendações são explicáveis. Cada indicação mostra quais atrações atendem quais interesses e quais classes foram inferidas.

O reasoner encontra erros de modelagem, como no teste de consistência.

A ontologia usa um padrão aberto e pode ser aberta no Protégé ou reutilizada por outro sistema.

**Limitações**

A ontologia não aprende com o comportamento dos usuários. Um sistema de filtragem colaborativa usaria as avaliações de outros viajantes, o que aqui não acontece.

O raciocínio é lento quando comparado a uma consulta em banco de dados. Cada recomendação executa o HermiT de novo, o que não escalaria para milhares de destinos.

A hipótese do mundo aberto dificulta regras com negação, como "nunca visitou".

A ordenação final ainda depende de uma pontuação com pesos escolhidos manualmente. OWL classifica bem, mas não ordena.

A base tem só dez destinos e os custos são aproximados.